# <center>**Chapter 22 - Visualizing a PyTorch Model**<center>

## **Mục tiêu**
- Huấn luyện một MLP phân loại Iris bằng PyTorch.
- Xuất mô hình sang định dạng ONNX với tên đầu vào/đầu ra rõ ràng.
- Kiểm tra tệp ONNX, sau đó mở bằng Netron để xem đồ thị tính toán.

`state_dict` chứa các tensor trọng số nhưng không mô tả cách các phép toán được nối với nhau. ONNX lưu đồ thị đã truy vết từ lần chạy forward nên có thể mở bằng Netron.

In [1]:
from pathlib import Path

import numpy as np
import torch
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data.astype(np.float32),
    iris.target.astype(np.int64),
    test_size=0.3,
    stratify=iris.target,
    random_state=SEED,
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=16,
    shuffle=True,
)

In [2]:
class IrisClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(4, 16)
        self.activation = nn.ReLU()
        self.output = nn.Linear(16, 3)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.output(self.activation(self.hidden(features)))

model = IrisClassifier().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train()
    for features, labels in train_loader:
        features, labels = features.to(DEVICE), labels.to(DEVICE)
        loss = loss_fn(model(features), labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

model.eval()
X_test_tensor = torch.from_numpy(X_test).to(DEVICE)
with torch.inference_mode():
    predictions = model(X_test_tensor).argmax(dim=1).cpu().numpy()
    
print(f"Accuracy Iris trên tập test: {accuracy_score(y_test, predictions):.3f}")
print("Kích thước đầu vào:", tuple(X_test_tensor[:1].shape))

Accuracy Iris trên tập test: 0.911
Kích thước đầu vào: (1, 4)


## **Xuất và kiểm tra mô hình ONNX**

Tensor mẫu phải có đúng số đặc trưng của mô hình. Đặt tên `features` và `logits` giúp nhận diện hai đầu mút trong sơ đồ; `dynamic_axes` cho phép thay đổi số mẫu trong một batch.

In [3]:
import onnx

def find_lab_code_dir() -> Path:
    lab_dir_name = "Buổi thực hành số 7 (Optimizing DL Networks)"
    current = Path.cwd().resolve()
    for root in (current, *current.parents):
        candidates = (root, root / "code", root / lab_dir_name / "code")
        for candidate in candidates:
            if candidate.is_dir() and candidate.name == "code" and candidate.parent.name == lab_dir_name:
                return candidate
    raise FileNotFoundError("Không tìm thấy thư mục code của buổi thực hành số 7.")

model.eval()
sample = X_test_tensor[:1]
onnx_path = find_lab_code_dir() / "iris_classifier.onnx"
torch.onnx.export(
    model,
    sample,
    str(onnx_path),
    input_names=["features"],
    output_names=["logits"],
    dynamic_axes={
        "features": {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
    opset_version=17,
    dynamo=False,
)

C:\Users\qhoan\AppData\Local\Temp\ipykernel_2400\1848571999.py:16: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [4]:
exported_model = onnx.load(str(onnx_path))
onnx.checker.check_model(exported_model)

print(f"Đã xuất và kiểm tra ONNX: {onnx_path.resolve()}")
print("Đầu vào:", [value.name for value in exported_model.graph.input])
print("Đầu ra:", [value.name for value in exported_model.graph.output])

Đã xuất và kiểm tra ONNX: D:\Programming\.vscode\Coding\Data Science\(!!!) 4.HK1 - Học sâu (Python)\Buổi thực hành số 7 (Optimizing DL Networks)\code\iris_classifier.onnx
Đầu vào: ['features']
Đầu ra: ['logits']


## **Xem đồ thị bằng Netron**

Mở `iris_classifier.onnx` bằng ứng dụng Netron trên máy tính (hoặc cài Netron và chạy `netron iris_classifier.onnx`). Đồ thị thể hiện đầu vào `features`, hai phép tuyến tính, ReLU và đầu ra `logits`. Có thể nhấp vào từng nút trong Netron để xem kích thước tensor và thông tin trọng số.

# <center>**HẾT**<center>